# STEREO EUVI region-of-interest example

Plot a helioprojective region from local STEREO EUVI FITS observations using SunPy.
Use the Miniforge `solarphysics_env_latest` environment and run all cells in order.
Input files remain local; set `EUVI_DATA_DIR` to choose a different folder. The default
is the 2025-01-24 STEREO-A 171 Å observation under `Local/observations/`.

The ROI and intensity limits below are example parameters, not calibration or event
selection criteria. PNG files are written only under ignored `Local/outputs/`;
rerunning replaces figures with the same input stem. This example makes no downloads.

See [the example guide](README.md) for setup and validation.

In [ ]:
import os
from pathlib import Path

import sunpy.map
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord

In [ ]:
# Run from the repository root or any directory beneath it.
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "Python/solar_toolkit").is_dir() and (p / "Apps").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from within the solarphysics repository.")

# Override with EUVI_DATA_DIR, or edit this parameter for your local FITS folder.
default_data_dir = repo_root / "Local/observations/stereo-a/euvi/20250124/171"
euvi_dir = Path(os.environ.get("EUVI_DATA_DIR", str(default_data_dir))).expanduser()
output_dir = repo_root / "Local/outputs/examples/stereo/euvi-roi"
if not euvi_dir.is_dir():
    raise FileNotFoundError("EUVI input directory is missing; set EUVI_DATA_DIR to your FITS folder.")
euvi_files = sorted(p for p in euvi_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in {".fts", ".fits", ".fit"})
if not euvi_files:
    raise FileNotFoundError("No FITS files found; check EUVI_DATA_DIR and the selected observation.")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Found {len(euvi_files)} FITS files; figures go to Local/outputs/examples/stereo/euvi-roi.")

In [ ]:
# Helioprojective coordinates in arcseconds, for the original 2025-01-24 example.
# Adjust these bounds and intensity limits for other observations.
roi_bottom_left = (-300, -500)
roi_top_right = (400, 200)
intensity_limits = (444, 4444)

In [ ]:
for i, file in enumerate(euvi_files):

    my_map = sunpy.map.Map(file)

    bottom_left = SkyCoord(
        roi_bottom_left[0] * u.arcsec,
        roi_bottom_left[1] * u.arcsec,
        frame=my_map.coordinate_frame
    )

    top_right = SkyCoord(
        roi_top_right[0] * u.arcsec,
        roi_top_right[1] * u.arcsec,
        frame=my_map.coordinate_frame
    )

    roi_map = my_map.submap(
        bottom_left,
        top_right=top_right
    )

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection=roi_map)

    roi_map.plot(
        axes=ax,
        vmin=intensity_limits[0],
        vmax=intensity_limits[1]
    )

    output_file = output_dir / f"{file.stem}_ROI.png"

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"[{i+1}/{len(euvi_files)}] Saved: {output_file.name}")